# Lab 10: Automated Multi‑Format Sales Report Pipeline and Delivery
### Course: Data Engineering / Data Processing Lab
### Objective
To design and implement a production‑ready automated report pipeline that extracts, transforms, and generates sales reports in multiple formats (Excel, HTML, PDF) and optionally delivers them via email.


## Problem Description
The objective of this lab is to build a data pipeline that:
- Loads sales data for multiple products across months.
- Calculates revenue for each record.
- Aggregates product level statistics.
- Generates reports in Excel, HTML, and PDF formats.
- Optionally sends the generated report via email.
- Stores reports in a folder with filenames including the current date.


## Step 1: Import Required Libraries


In [ ]:
import pandas as pd
import os
from datetime import datetime
from jinja2 import Template
from fpdf import FPDF
import smtplib
from email.message import EmailMessage
import matplotlib.pyplot as plt


## Step 2: Extract Phase (Load Sales Data)
In this step we create or load a dataset containing monthly sales data.


In [ ]:
data = {
    'Month': ['Jan','Jan','Jan','Feb','Feb','Feb','Mar','Mar','Mar'],
    'Product': ['Laptop','Phone','Tablet','Laptop','Phone','Tablet','Laptop','Phone','Tablet'],
    'Units Sold': [50,120,70,65,140,80,75,160,90],
    'Unit Price': [800,500,300,800,500,300,800,500,300],
    'Region': ['East','West','North','East','West','North','East','West','North']
}

df = pd.DataFrame(data)
df.head()

## Step 3: Transform Phase
We calculate revenue and generate summary statistics.


In [ ]:
# Calculate Revenue
df['Revenue'] = df['Units Sold'] * df['Unit Price']

# Product Summary
summary = df.groupby('Product').agg(
    Total_Units=('Units Sold','sum'),
    Total_Revenue=('Revenue','sum'),
    Avg_Price=('Unit Price','mean')
).reset_index()

total_revenue = summary['Total_Revenue'].sum()
summary['Revenue Share (%)'] = (summary['Total_Revenue']/total_revenue)*100

summary

## Monthly Revenue Calculation


In [ ]:
monthly_revenue = df.groupby('Month')['Revenue'].sum().reset_index()
monthly_revenue

## Step 4: Generate Excel Report


In [ ]:
os.makedirs('reports', exist_ok=True)
date_str = datetime.now().strftime('%Y-%m-%d')

excel_file = f'reports/sales_report_{date_str}.xlsx'

with pd.ExcelWriter(excel_file) as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    monthly_revenue.to_excel(writer, sheet_name='Monthly Revenue', index=False)

excel_file

## Step 5: Generate HTML Report Using Jinja2


In [ ]:
template_html = """
<html>
<head>
<title>Sales Report</title>
<style>
table {border-collapse: collapse; width: 80%;}
th, td {border:1px solid black; padding:8px; text-align:center;}
th {background-color:#dddddd;}
</style>
</head>
<body>
<h2>Sales Summary</h2>
{{ table }}
<h3>Total Revenue: {{ total }}</h3>
</body>
</html>
"""

template = Template(template_html)

html_content = template.render(
    table=summary.to_html(index=False),
    total=total_revenue
)

html_file = f'reports/sales_report_{date_str}.html'

with open(html_file,'w') as f:
    f.write(html_content)

html_file

## Step 6: Generate PDF Report Using FPDF


In [ ]:
pdf = FPDF()
pdf.add_page()
pdf.set_font('Arial','B',16)
pdf.cell(200,10,'Sales Report',ln=True,align='C')

pdf.set_font('Arial','B',12)

for col in summary.columns:
    pdf.cell(40,10,col,1)
pdf.ln()

pdf.set_font('Arial','',12)

for row in summary.values:
    for item in row:
        pdf.cell(40,10,str(round(item,2)),1)
    pdf.ln()

pdf_file = f'reports/sales_report_{date_str}.pdf'
pdf.output(pdf_file)

pdf_file

## Step 7: Add Bar Chart of Monthly Revenue (for PDF or analysis)


In [ ]:
plt.figure()
plt.bar(monthly_revenue['Month'], monthly_revenue['Revenue'])
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.title('Monthly Revenue')
plt.savefig('reports/monthly_revenue_chart.png')
plt.show()

## Step 8: Email Delivery (SMTP Example)


In [ ]:
def send_email(sender, password, receiver, file_path):
    msg = EmailMessage()
    msg['Subject'] = 'Automated Sales Report'
    msg['From'] = sender
    msg['To'] = receiver
    msg.set_content('Please find the attached sales report.')

    with open(file_path,'rb') as f:
        msg.add_attachment(f.read(), maintype='application', subtype='octet-stream', filename=os.path.basename(file_path))

    with smtplib.SMTP_SSL('smtp.gmail.com',465) as smtp:
        smtp.login(sender,password)
        smtp.send_message(msg)

# Example usage
# send_email('your_email@gmail.com','app_password','receiver@gmail.com', pdf_file)

# Short Report

## Role of Each Pipeline Step

**Extract:**
The extract phase loads raw sales data from a source such as CSV files, databases, or APIs. In this lab, a dataset containing product sales information across months is loaded into a Pandas DataFrame.

**Transform:**
In this step, the raw data is processed to generate useful insights. Revenue is calculated, product summaries are generated, and monthly revenue totals are computed.

**Generate:**
The processed data is converted into different report formats such as Excel, HTML, and PDF. These reports contain structured tables and formatted summaries that are easy to interpret.

**Deliver:**
Finally, the generated report is distributed to stakeholders. In this pipeline, delivery is implemented through email using SMTP.

## Ensuring Report Consistency
All reports (Excel, HTML, and PDF) are generated from the same transformed dataset (summary and monthly revenue tables). Because the reports are derived from the same processed DataFrame, the information remains consistent across formats.

## Adding Bar Chart to PDF
A bar chart representing monthly revenue can be generated using Matplotlib. The chart is saved as an image and embedded into the PDF report so that readers can visually analyze revenue trends.

## Securing Email Credentials in Production
Email credentials should never be hardcoded in the script. Instead they should be secured using environment variables, secret managers, or encrypted configuration files. Cloud platforms such as AWS Secrets Manager or Google Secret Manager can also be used to safely store credentials.
